# Figure 1 — data preparation

Builds every input the Figure 1 R scripts read, into `figure_1/data/`. Run this before
`Figure_1_combined.R`.

**Why this notebook exists.** `manh_plot_dataprep.ipynb` reads and writes absolute paths under
`/Users/polina/genetics_gsea/data/`, so it cannot run in this repo, and three of the files the R
scripts expect were never committed:

| File the R scripts read | Status before this notebook |
| ----------------------- | --------------------------- |
| `data/qd_sl_eff.csv`, `data/qm_sl_eff.csv` (panel b) | missing from the repo entirely |
| `data/disease_ta_measur_index.snappy.parquet` (panel d) | missing from the repo entirely |
| `data/disease_ta_index_pandas.csv` (panel d gene labels) | present, but one directory up |
| `data/l2g_diseases_full.csv` (panel c) | present, but one directory up |

`Figure_1_b_c.R` guards the panel b inputs with `if (file.exists(...))`, so the missing CSVs made it
silently emit a two-panel figure, and `Figure_1_combined.R` then crashed on a `grobs_list` that only
exists in the four-panel branch.

**Panel d provenance.** `manh_plot_dataprep.ipynb` built its parquet as

```python
disease_ta_measur       = disease_ta.join(measur, on="geneId", how="outer")
disease_ta_measur_index = disease_ta_measur.join(target_index, on="geneId", how="inner")
```

`disease_ta` and `target_index` are both committed (`figure_1/genes_therapeutic_areas/`,
`figure_1/target_index_for_plot.parquet/`). Only `measur` — unique measurement traits per gene —
was missing, and it is recomputed here from the R1 measurement table. That column feeds a
circos track which is **commented out** in `Figure_1_d.R`, and measurement-only genes entering
through the outer join carry NA on both plotted tracks, so this recomputation cannot change the
rendered panel.

**Panel c.** Instead of shipping the 16 MB row-level CSV and recomputing cumulative curves in R
(which required splitting a Python list-repr string on commas), this notebook writes the finished
per-year layer heights. They come straight from
`data/intermediate_files/fig1c_cumulative_discovery_nested-r1.csv`, so panel c, ED Fig 3 and the
reviewer response are guaranteed to agree.

**Donut in panel d.** Slice counts come from `study_ancestry_classification-r1.csv` rather than
being hardcoded in the R script.

## Setup

In [1]:
import shutil
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from pyspark.sql import functions as f

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/10 16:32:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
fig1_dir = Path(".").resolve()
repo_root = fig1_dir.parents[2]

path_to_release_folder = repo_root / "data" / "25.06"
path_to_intermediate_data_folder = repo_root / "data" / "intermediate_files"

out_dir = fig1_dir / "data"
out_dir.mkdir(exist_ok=True)

MAX_YEAR = 2024
print(f"figure_1 dir: {fig1_dir}")
print(f"output dir:   {out_dir}")

figure_1 dir: /Users/yt4/Projects/Gentropy-manuscript/chapters/03-manuscript-figures/figure_1
output dir:   /Users/yt4/Projects/Gentropy-manuscript/chapters/03-manuscript-figures/figure_1/data


## 1. Panel b — `qd_sl_eff.csv` and `qm_sl_eff.csv`

Reproduces the "Temporal dependences of studies" cells of
`chapters/02-analysis/01-descriptions-numbers/02_gene_stauration_plots.ipynb`: lead-variant effects
joined to the study year and sample size, split into qualifying disease and measurement credible
sets.

One deviation, forced by the data: that notebook read a `25.07` snapshot straight from GCS and used
`abs(rescaledStatistics.estimatedBeta)`. The local `lead_variant_effect` table has no
`estimatedBeta` field — `rescaledStatistics` exposes `absEstimatedBeta` — so that is used here, and
the output column keeps the name `absEstimatedBeta` that the R script reads. Panel b's lines may
therefore differ marginally from the published version.

In [4]:
si = StudyIndex.from_parquet(session, str(path_to_release_folder / "output/study"))

# FinnGen R12 carries no publication date in the release; the original analysis pins it to the
# R12 release date.
si_year = (
    si.df.withColumn(
        "publicationDate",
        f.when(f.col("projectId") == "FINNGEN_R12", f.lit("2024-11-04")).otherwise(f.col("publicationDate")),
    )
    .withColumn("year", f.col("publicationDate").substr(1, 4).cast("int"))
    .select("studyId", "year", "nSamples")
)

sl_eff = session.spark.read.parquet(str(path_to_intermediate_data_folder / "lead_variant_effect"))
sl_eff_slim = (
    sl_eff.select(
        "studyId",
        "studyLocusId",
        f.col("rescaledStatistics.absEstimatedBeta").alias("absEstimatedBeta"),
    )
    .join(si_year, on="studyId", how="inner")
    .cache()
)
print(f"lead variant effect rows with year: {sl_eff_slim.count():,}")

lead variant effect rows with year: 2,833,758


In [5]:
qd_cs = (
    session.spark.read.parquet(str(path_to_intermediate_data_folder / "qualifying_credible_sets"))
    .select("studyLocusId")
    .distinct()
)
qm_cs = (
    session.spark.read.parquet(str(path_to_intermediate_data_folder / "qualifying_measurement_credible_sets"))
    .select("studyLocusId")
    .distinct()
)

qd_sl_eff = sl_eff_slim.join(qd_cs, on="studyLocusId", how="inner").toPandas()
qm_sl_eff = sl_eff_slim.join(qm_cs, on="studyLocusId", how="inner").toPandas()

qd_sl_eff.to_csv(out_dir / "qd_sl_eff.csv", index=False)
qm_sl_eff.to_csv(out_dir / "qm_sl_eff.csv", index=False)
print(f"qd_sl_eff.csv: {len(qd_sl_eff):,} rows")
print(f"qm_sl_eff.csv: {len(qm_sl_eff):,} rows")
qd_sl_eff.head(3)

qd_sl_eff.csv: 70,618 rows
qm_sl_eff.csv: 450,357 rows


,studyLocusId,studyId,absEstimatedBeta,year,nSamples
0,13575c752b770a2ce8132a869d70fb89,GCST90302234,0.083234,2023,96432
1,0439c9fcb5b9f1bcfe028f50be1c0a2c,GCST011048,0.045722,2021,177526
2,2f1074a06266047cb2ac107761cf7f07,GCST90435597,0.163193,2018,406220


## 2. Panel c — per-year stacked layer heights

Nesting order, bottom to top: EUR common, non-EUR common, mixed common, rare (any ancestry). Each
layer is the increment that tier adds over the tiers below it, exactly as computed in
`chapters/06-review-r1/ancestry-mixed-split/01_ancestry_reclassification.ipynb`.

In [6]:
nested = pd.read_csv(path_to_intermediate_data_folder / "fig1c_cumulative_discovery_nested-r1.csv")

fig1c_layers = (
    nested[nested["metric"].isin(["disease genes", "gene-disease pairs"])]
    .loc[:, ["metric", "year", "tier_index", "layer_label", "cumulative", "layer"]]
    .sort_values(["metric", "year", "tier_index"])
    .reset_index(drop=True)
)
fig1c_layers.to_csv(out_dir / "fig1c_layers-r1.csv", index=False)
print(f"fig1c_layers-r1.csv: {len(fig1c_layers):,} rows")

# Final-year totals, for eyeballing against the figure.
fig1c_layers[fig1c_layers["year"] == MAX_YEAR].pivot_table(
    index="layer_label", columns="metric", values=["layer", "cumulative"]
)

fig1c_layers-r1.csv: 152 rows


cumulative                            layer  \
metric         disease genes gene-disease pairs disease genes   
layer_label                                                     
EUR common            5552.0            18521.0        5552.0   
mixed common          8014.0            34905.0         633.0   
non-EUR common        7381.0            30806.0        1829.0   
rare                  8129.0            35535.0         115.0   

                                   
metric         gene-disease pairs  
layer_label                        
EUR common                18521.0  
mixed common               4099.0  
non-EUR common            12285.0  
rare                        630.0

## 3. Panel d — `disease_ta_measur_index.snappy.parquet`

Written as a single Parquet **file**, not a Spark output directory, because `Figure_1_d.R` calls
`arrow::read_parquet()` on that exact filename.

In [7]:
disease_ta = session.spark.read.parquet(str(fig1_dir / "genes_therapeutic_areas"))
target_index = (
    session.spark.read.parquet(str(fig1_dir / "target_index_for_plot.parquet"))
    .withColumnRenamed("targetId", "geneId")
    .drop("approvedSymbol")
)
print(f"disease_ta genes:   {disease_ta.count():,}")
print(f"target_index genes: {target_index.count():,}")

disease_ta genes:   8,285
target_index genes: 20,083


In [8]:
# `measur`: unique measurement traits per gene, recomputed from the R1 row-level table.
l2g_ancestry = session.spark.read.parquet(
    str(path_to_intermediate_data_folder / "list_of_prioritised_genes_per_CS_with_year_ancestry-r1.parquet")
)
measur = (
    l2g_ancestry.join(qm_cs, on="studyLocusId", how="inner")
    .withColumn("traitId", f.explode("diseaseIds"))
    .groupBy("geneId")
    .agg(f.countDistinct("traitId").alias("uniqueMeasurement"))
)
print(f"genes with measurement associations: {measur.count():,}")

genes with measurement associations: 15,160


In [9]:
disease_ta_measur_index = (
    disease_ta.join(measur, on="geneId", how="outer").join(target_index, on="geneId", how="inner").toPandas()
)
print(f"disease_ta_measur_index rows: {len(disease_ta_measur_index):,}")
print(f"  with uniqueDiseases:        {disease_ta_measur_index['uniqueDiseases'].notna().sum():,}")
print(f"  with uniqueMeasurement:     {disease_ta_measur_index['uniqueMeasurement'].notna().sum():,}")

pq.write_table(
    pa.Table.from_pandas(disease_ta_measur_index, preserve_index=False),
    out_dir / "disease_ta_measur_index.snappy.parquet",
    compression="snappy",
)
print(f"wrote {out_dir / 'disease_ta_measur_index.snappy.parquet'}")

26/08/10 16:33:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


disease_ta_measur_index rows: 15,641
  with uniqueDiseases:        8,285
  with uniqueMeasurement:     15,160
wrote /Users/yt4/Projects/Gentropy-manuscript/chapters/03-manuscript-figures/figure_1/data/disease_ta_measur_index.snappy.parquet


In [10]:
# Gene labels in panel d are read from data/disease_ta_index_pandas.csv; the committed copy sits
# one directory up.
shutil.copyfile(fig1_dir / "disease_ta_index_pandas.csv", out_dir / "disease_ta_index_pandas.csv")
print(f"copied disease_ta_index_pandas.csv into {out_dir}")

copied disease_ta_index_pandas.csv into /Users/yt4/Projects/Gentropy-manuscript/chapters/03-manuscript-figures/figure_1/data


## 4. Donut in panel d — ancestry slice counts

Five slices. `EUR` and `mixed` are the new ancestry classes; `AFR` and `EAS/CSA` are the non-EUR
studies whose predominant ancestry is `afr` and `eas`; `Other` is everything left over, i.e.
non-EUR studies that are predominantly `amr` or Finnish.

The published donut had four slices (EUR / AFR / EAS-CSA / Other) with hardcoded counts, which hid
all 11,725 pan-ancestry studies inside `Other`.

In [11]:
study_ancestry = pd.read_csv(path_to_intermediate_data_folder / "study_ancestry_classification-r1.csv")
gwas = study_ancestry[study_ancestry["studyType"] == "gwas"]

slices = {
    "EUR": (gwas["ancestryClass"] == "EUR").sum(),
    "AFR": ((gwas["ancestryClass"] == "non-EUR") & (gwas["predominantAncestry"] == "afr")).sum(),
    "EAS/CSA": ((gwas["ancestryClass"] == "non-EUR") & (gwas["predominantAncestry"] == "eas")).sum(),
    "mixed": (gwas["ancestryClass"] == "mixed").sum(),
}
slices["Other"] = len(gwas) - sum(slices.values())

donut = pd.DataFrame({"ancestry": list(slices), "value": list(slices.values())})
donut["percent"] = (100 * donut["value"] / donut["value"].sum()).round(2)
assert donut["value"].sum() == len(gwas), "slices must partition the GWAS studies"

donut.to_csv(out_dir / "ancestry_donut_counts-r1.csv", index=False)
print(f"total GWAS studies: {len(gwas):,}")
donut

total GWAS studies: 100,526


,ancestry,value,percent
0,EUR,65253,64.91
1,AFR,7421,7.38
2,EAS/CSA,11785,11.72
3,mixed,11725,11.66
4,Other,4342,4.32


In [12]:
# What "Other" is made of, so the caption can state it.
other = gwas[(gwas["ancestryClass"] == "non-EUR") & (~gwas["predominantAncestry"].isin(["afr", "eas"]))]
other["predominantAncestry"].value_counts().rename_axis("predominantAncestry").reset_index(name="n_studies")

,predominantAncestry,n_studies
0,fin,2303
1,amr,2039


## Files written

Into `chapters/03-manuscript-figures/figure_1/data/`:

| File | Used by |
| ---- | ------- |
| `qd_sl_eff.csv`, `qm_sl_eff.csv` | panel b — average sample size, average \|β\| |
| `fig1c_layers-r1.csv` | panel c — stacked bars |
| `disease_ta_measur_index.snappy.parquet` | panel d — circular Manhattan |
| `disease_ta_index_pandas.csv` | panel d — gene labels |
| `ancestry_donut_counts-r1.csv` | panel d — centre donut |